# Manipulation der eps-real-Ableitung

Dieses Notebook baut auf der Interpolation auf und entfernt Ausreißer erst in `Eps_real_derivative`. Dadurch bleiben konstante Offsets in Teilen eines Spektrums weitgehend unschädlich; markiert werden vor allem die künstlichen Ableitungspunkte an Offset-Rändern.

## Einstellungen

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9, 6),
    "axes.grid": True,
})

COLUMNS = [
    "Freq_Hz",
    "Time_s",
    "Eps_real",
    "Eps_imag",
    "Temp_K",
    "MTime_s",
    "Phi_deg",
    "Z_real_Ohm",
    "Z_imag_Ohm",
    "TanPhi",
]

TIME_COL = "Time_Relative_s"
INTERPOLATED_VALUE_COLS = ("Eps_real",)

MATERIAL = "PEI5mgmL"
TEMPERATURE = "80°C"
MODE = "Abs"
PLOT_TIME_S = 4000

# None lädt alle PEI5mgmL-Dateien. Für eine einzelne Serie z.B. f"{MATERIAL}_{TEMPERATURE}_{MODE}.TXT" setzen.
SELECTED_FILE = None
MATERIAL_FILTER = MATERIAL

# Filterparameter für die Ableitungsspektren.
REQUIRE_POSITIVE_DERIVATIVE = True
OUTLIER_WINDOW = 5
OUTLIER_THRESHOLD_MAD = 4.5
OUTLIER_MIN_PERIODS = 3
EDGE_POINTS_TO_IGNORE_FOR_ROLLING = 0

# Optionaler Export der bereinigten Ableitung.
EXPORT_CLEAN_DERIVATIVE = False
EXPORT_DIR = Path("data") / "manipulated"


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data" / "data_raw").exists():
            return path
    raise FileNotFoundError("Could not find the project root with data/data_raw.")


def add_code_dir_to_path(project_root):
    code_dir = project_root / "Code"
    code_dir_text = str(code_dir)
    if code_dir_text not in sys.path:
        sys.path.insert(0, code_dir_text)


PROJECT_ROOT = find_project_root()
add_code_dir_to_path(PROJECT_ROOT)
import switch_points
importlib.reload(switch_points)
manual_switch_points = switch_points.MANUAL_SWITCH_POINTS

DATA_DIR = PROJECT_ROOT / "data" / "data_raw"
DATA_DIR

## Laden und relative Schaltzeit

In [ ]:
def load_measurements(data_dir=DATA_DIR, selected_file=None, material_filter="PEI5mgmL"):
    if selected_file is None:
        file_paths = sorted(data_dir.glob("*.TXT")) + sorted(data_dir.glob("*.txt"))
    else:
        file_path = data_dir / selected_file
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        file_paths = [file_path]

    if selected_file is None and material_filter is not None:
        file_paths = [path for path in file_paths if path.stem.split("_")[0] == material_filter]

    if not file_paths:
        raise FileNotFoundError(f"No .TXT files found in: {data_dir}")

    frames = []
    for path in file_paths:
        df = pd.read_csv(
            path,
            sep=r"\s+",
            skiprows=4,
            names=COLUMNS,
            encoding="latin1",
            engine="python",
        )

        parts = path.stem.split("_")
        material, temperature, mode = parts[:3] if len(parts) >= 3 else (path.stem, None, None)
        first_freq = df["Freq_Hz"].iloc[0]
        df["Spectrum_ID"] = (df["Freq_Hz"] == first_freq).cumsum() - 1
        df["Spectrum_Number"] = df["Spectrum_ID"] + 1
        df["Point_Number"] = range(1, len(df) + 1)
        df["Material"] = material
        df["Temperature"] = temperature
        df["Mode"] = mode
        df["Source_File"] = path.name
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


def add_relative_switch_time(df, switch_points):
    df = df.copy()
    switch_time_rows = []

    for dataset_key, dataset in df.groupby(["Material", "Temperature", "Mode"], dropna=False):
        switch_point_number = switch_points.get(dataset_key)
        if switch_point_number is None:
            raise ValueError(f"Missing Switch_Point_Number for series: {dataset_key}")

        switch_point = dataset[dataset["Point_Number"] == switch_point_number]
        next_point = dataset[dataset["Point_Number"] == switch_point_number + 1]
        if switch_point.empty or next_point.empty:
            raise ValueError(f"Switch-point pair not found: {dataset_key}, {switch_point_number}")

        switch_time_rows.append({
            "Material": dataset_key[0],
            "Temperature": dataset_key[1],
            "Mode": dataset_key[2],
            "Switch_Point_Number": int(switch_point_number),
            "Switch_Spectrum_ID": int(switch_point["Spectrum_ID"].iloc[0]),
            "Switch_Time_s": float((switch_point["MTime_s"].iloc[0] + next_point["MTime_s"].iloc[0]) / 2),
        })

    switch_times = pd.DataFrame(switch_time_rows)
    df = df.merge(switch_times, on=["Material", "Temperature", "Mode"], how="left")
    df[TIME_COL] = df["MTime_s"] - df["Switch_Time_s"]
    return df, switch_times


df_all = load_measurements(selected_file=SELECTED_FILE, material_filter=MATERIAL_FILTER)
df_all, switch_times = add_relative_switch_time(df_all, manual_switch_points)

print(f"Loaded rows: {len(df_all):,}")
print(switch_times.to_string(index=False))

## Interpolation wie im Interpolationsnotebook

In [ ]:
def _linear_at(target_time, times, values):
    if len(times) < 2:
        return np.nan
    x0, x1 = float(times[0]), float(times[1])
    y0, y1 = float(values[0]), float(values[1])
    if x0 == x1:
        return np.nan
    return y0 + (target_time - x0) * (y1 - y0) / (x1 - x0)


def _insert_switch_support_point(freq_data, value_col, time_col=TIME_COL, target_time=0):
    before = freq_data[freq_data[time_col] < target_time].tail(2)
    support_points = freq_data[[time_col, value_col]].dropna().copy()
    support_points = support_points[~np.isclose(support_points[time_col], target_time)]

    if len(before) >= 2:
        switch_value = _linear_at(target_time, before[time_col].to_numpy(), before[value_col].to_numpy())
        support_points = pd.concat(
            [support_points, pd.DataFrame({time_col: [float(target_time)], value_col: [switch_value]})],
            ignore_index=True,
        )

    support_points = support_points.sort_values(time_col)
    return support_points.groupby(time_col, as_index=False)[value_col].mean()


def interpolate_complete_spectra(
    df,
    time_col=TIME_COL,
    freq_col="Freq_Hz",
    value_cols=INTERPOLATED_VALUE_COLS,
    dataset_cols=("Source_File", "Material", "Temperature", "Mode"),
    drop_incomplete=True,
    include_switch_time=True,
):
    rows = []
    metadata_cols = ["Switch_Point_Number", "Switch_Spectrum_ID", "Switch_Time_s"]

    for dataset_key, dataset in df.groupby(list(dataset_cols), dropna=False):
        dataset_key = dataset_key if isinstance(dataset_key, tuple) else (dataset_key,)
        dataset_meta = dict(zip(dataset_cols, dataset_key))
        target_times = np.sort(dataset[time_col].dropna().unique())
        if include_switch_time and not np.isclose(target_times, 0).any():
            target_times = np.sort(np.append(target_times, 0.0))
        frequencies = np.sort(dataset[freq_col].dropna().unique())
        dataset_metadata = {
            col: dataset[col].dropna().iloc[0]
            for col in metadata_cols
            if col in dataset.columns and not dataset[col].dropna().empty
        }

        interpolated_by_freq = {}
        for freq in frequencies:
            freq_data = dataset.loc[dataset[freq_col] == freq, [time_col, *value_cols]].dropna(subset=[time_col])
            freq_data = freq_data.sort_values(time_col).groupby(time_col, as_index=False)[list(value_cols)].mean()

            interpolated_values = {}
            for value_col in value_cols:
                support = _insert_switch_support_point(freq_data, value_col, time_col=time_col, target_time=0)
                interpolated_values[value_col] = np.interp(
                    target_times,
                    support[time_col].to_numpy(),
                    support[value_col].to_numpy(),
                    left=np.nan,
                    right=np.nan,
                )
            interpolated_by_freq[freq] = interpolated_values

        for spectrum_id, target_time in enumerate(target_times):
            for freq in frequencies:
                row = {
                    **dataset_meta,
                    **dataset_metadata,
                    "Interpolated_Spectrum_ID": spectrum_id,
                    time_col: target_time,
                    freq_col: freq,
                    "Is_Switch_Spectrum": bool(np.isclose(target_time, 0)),
                }
                for value_col in value_cols:
                    row[value_col] = interpolated_by_freq[freq][value_col][spectrum_id]
                rows.append(row)

    interpolated = pd.DataFrame(rows)
    if drop_incomplete:
        complete_ids = [*dataset_cols, "Interpolated_Spectrum_ID"]
        complete_mask = interpolated.groupby(complete_ids, dropna=False)[list(value_cols)].transform(
            lambda values: values.notna().all()
        )
        interpolated = interpolated[complete_mask.all(axis=1)].reset_index(drop=True)
    return interpolated


df_interpolated_complete = interpolate_complete_spectra(df_all)
print(f"Interpolated rows: {len(df_interpolated_complete):,}")

## Ableitung berechnen

In [ ]:
def derive_eps_real_by_frequency(
    df,
    value_col="Eps_real",
    freq_col="Freq_Hz",
    time_col=TIME_COL,
    spectrum_col="Interpolated_Spectrum_ID",
    dataset_cols=("Source_File", "Material", "Temperature", "Mode"),
):
    rows = []
    group_cols = [*dataset_cols, spectrum_col, time_col]

    for group_key, spectrum in df.groupby(group_cols, dropna=False):
        spectrum = spectrum.sort_values(freq_col).reset_index(drop=True)
        if len(spectrum) < 2:
            continue

        frequencies = spectrum[freq_col].to_numpy()
        eps_real = spectrum[value_col].to_numpy()
        omega = 2 * np.pi * frequencies
        ln_omega = np.log(omega)
        derivative = -np.pi / 2 * np.diff(eps_real) / np.diff(ln_omega)
        omega_mid = np.exp((ln_omega[:-1] + ln_omega[1:]) / 2)
        freq_mid = omega_mid / (2 * np.pi)

        group_key = group_key if isinstance(group_key, tuple) else (group_key,)
        meta = dict(zip(group_cols, group_key))
        for i, (freq, omega_value, derivative_value) in enumerate(zip(freq_mid, omega_mid, derivative)):
            rows.append({
                **meta,
                "Derivative_Point_Index": i,
                "Freq_Hz_left": frequencies[i],
                "Freq_Hz_right": frequencies[i + 1],
                "Eps_real_left": eps_real[i],
                "Eps_real_right": eps_real[i + 1],
                "Freq_Hz_mid": freq,
                "Omega_rad_s": omega_value,
                "Eps_real_derivative": derivative_value,
            })

    return pd.DataFrame(rows)


df_derivative = derive_eps_real_by_frequency(df_interpolated_complete)
print(f"Derivative rows: {len(df_derivative):,}")
df_derivative.head()

## Automatische Ausreißermarkierung in der Ableitung

In [ ]:
def _rolling_median_mad(values, window=5, min_periods=3):
    values = pd.Series(values, dtype="float64")
    median = values.rolling(window=window, center=True, min_periods=min_periods).median()
    deviation = (values - median).abs()
    mad = deviation.rolling(window=window, center=True, min_periods=min_periods).median()
    sigma = 1.4826 * mad

    valid_sigma = sigma[np.isfinite(sigma) & (sigma > 0)]
    fallback = float(valid_sigma.median()) if len(valid_sigma) else np.nan
    if not np.isfinite(fallback) or fallback <= 0:
        valid_values = values[np.isfinite(values)]
        fallback = float(valid_values.std()) if len(valid_values) else np.nan
    if not np.isfinite(fallback) or fallback <= 0:
        fallback = 1e-12

    sigma = sigma.replace(0, np.nan).fillna(fallback)
    return median, sigma


def mark_derivative_outliers(
    df,
    value_col="Eps_real_derivative",
    group_cols=("Source_File", "Material", "Temperature", "Mode", "Interpolated_Spectrum_ID", TIME_COL),
    window=OUTLIER_WINDOW,
    threshold=OUTLIER_THRESHOLD_MAD,
    min_periods=OUTLIER_MIN_PERIODS,
    require_positive=REQUIRE_POSITIVE_DERIVATIVE,
    edge_points_to_ignore=EDGE_POINTS_TO_IGNORE_FOR_ROLLING,
):
    marked = df.copy()
    marked["Derivative_Outlier"] = False
    marked["Derivative_Outlier_Reason"] = ""
    marked["Derivative_Log_Residual"] = np.nan
    marked["Derivative_Robust_Z"] = np.nan

    for _, spectrum in marked.groupby(list(group_cols), dropna=False, sort=False):
        spectrum = spectrum.sort_values("Omega_rad_s")
        index = spectrum.index
        values = spectrum[value_col].astype(float)

        invalid = ~np.isfinite(values)
        if require_positive:
            invalid |= values <= 0

        positive_values = values.where(values > 0)
        log_values = np.log(positive_values)
        median, sigma = _rolling_median_mad(log_values.to_numpy(), window=window, min_periods=min_periods)
        residual = log_values.reset_index(drop=True) - median
        robust_z = residual.abs() / sigma
        local_outlier = robust_z > threshold

        if edge_points_to_ignore > 0 and len(local_outlier) > 2 * edge_points_to_ignore:
            local_outlier.iloc[:edge_points_to_ignore] = False
            local_outlier.iloc[-edge_points_to_ignore:] = False

        marked.loc[index, "Derivative_Log_Residual"] = residual.to_numpy()
        marked.loc[index, "Derivative_Robust_Z"] = robust_z.to_numpy()

        invalid_index = index[invalid.to_numpy()]
        outlier_index = index[local_outlier.fillna(False).to_numpy()]
        marked.loc[invalid_index, "Derivative_Outlier"] = True
        marked.loc[invalid_index, "Derivative_Outlier_Reason"] = "nonpositive_or_nonfinite"
        marked.loc[outlier_index, "Derivative_Outlier"] = True
        marked.loc[outlier_index, "Derivative_Outlier_Reason"] = marked.loc[outlier_index, "Derivative_Outlier_Reason"].replace("", "local_log_mad")

    return marked


df_derivative_marked = mark_derivative_outliers(df_derivative)
df_derivative_clean = df_derivative_marked[~df_derivative_marked["Derivative_Outlier"]].copy()
df_derivative_outliers = df_derivative_marked[df_derivative_marked["Derivative_Outlier"]].copy()

print(f"Derivative rows before: {len(df_derivative_marked):,}")
print(f"Marked outliers: {len(df_derivative_outliers):,}")
print(f"Derivative rows after: {len(df_derivative_clean):,}")
df_derivative_outliers.head()

## Zusammenfassung der entfernten Punkte

In [ ]:
outlier_summary = (
    df_derivative_marked
    .groupby(["Source_File", "Material", "Temperature", "Mode"], dropna=False)
    .agg(
        derivative_points=("Eps_real_derivative", "size"),
        outliers=("Derivative_Outlier", "sum"),
        spectra=("Interpolated_Spectrum_ID", "nunique"),
    )
    .reset_index()
)
outlier_summary["outlier_fraction_percent"] = 100 * outlier_summary["outliers"] / outlier_summary["derivative_points"]
outlier_summary.sort_values(["Temperature", "Mode"]).style.format({"outlier_fraction_percent": "{:.2f}"})

## Kontrolle am einzelnen Spektrum

In [ ]:
def nearest_available_time(df, target_time, time_col=TIME_COL):
    available_times = np.sort(df[time_col].dropna().unique())
    if len(available_times) == 0:
        raise ValueError("No available times found.")
    return available_times[np.abs(available_times - target_time).argmin()]


def filter_series(df, material=MATERIAL, temperature=TEMPERATURE, mode=MODE):
    return df[(df["Material"] == material) & (df["Temperature"] == temperature) & (df["Mode"] == mode)].copy()


def plot_derivative_manipulation(
    df_marked,
    material=MATERIAL,
    temperature=TEMPERATURE,
    mode=MODE,
    target_time=PLOT_TIME_S,
):
    plot_base = filter_series(df_marked, material, temperature, mode)
    if plot_base.empty:
        raise ValueError("No derivative data found for this selection.")

    plot_time = nearest_available_time(plot_base, target_time)
    plot_df = plot_base[plot_base[TIME_COL] == plot_time].sort_values("Omega_rad_s")
    keep = plot_df[~plot_df["Derivative_Outlier"]]
    outliers = plot_df[plot_df["Derivative_Outlier"]]

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(plot_df["Omega_rad_s"], plot_df["Eps_real_derivative"], color="0.75", linewidth=1, label="original")
    ax.scatter(keep["Omega_rad_s"], keep["Eps_real_derivative"], color="black", s=24, label="used")
    ax.scatter(outliers["Omega_rad_s"], outliers["Eps_real_derivative"], color="red", s=42, marker="x", label="removed")

    ax.set_xscale("log")
    if (plot_df["Eps_real_derivative"] > 0).all():
        ax.set_yscale("log")
    else:
        ax.set_yscale("symlog", linthresh=1)
    ax.set_xlabel("Angular frequency omega (rad/s)")
    ax.set_ylabel("Eps real derivative")
    ax.set_title(f"Derivative manipulation at t_rel = {plot_time:.2f} s | {material} {temperature} {mode}")
    ax.grid(True, which="both")
    ax.legend()
    plt.tight_layout()
    plt.show()

    return plot_df


plot_df = plot_derivative_manipulation(df_derivative_marked)
plot_df[plot_df["Derivative_Outlier"]]

## Interaktive Kontrolle

In [ ]:
def interactive_derivative_manipulation(
    df_marked,
    material=MATERIAL,
    temperature=TEMPERATURE,
    mode=MODE,
    initial_time=PLOT_TIME_S,
):
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError as exc:
        raise ImportError("Install ipywidgets to use the interactive derivative slider.") from exc

    plot_base = filter_series(df_marked, material, temperature, mode)
    if plot_base.empty:
        raise ValueError("No derivative data found for this selection.")

    available_times = np.sort(plot_base[TIME_COL].dropna().unique())
    initial_available_time = nearest_available_time(plot_base, initial_time)
    slider_options = [(f"{time / 60:.2f} min ({time:.1f} s)", float(time)) for time in available_times]

    time_slider = widgets.SelectionSlider(
        options=slider_options,
        value=float(initial_available_time),
        description="t_rel",
        continuous_update=False,
        layout=widgets.Layout(width="95%"),
        style={"description_width": "60px"},
    )
    output = widgets.Output()
    controls = widgets.VBox([time_slider, output])

    def redraw(change=None):
        with output:
            output.clear_output(wait=True)
            display(plot_derivative_manipulation(df_marked, material, temperature, mode, time_slider.value))

    time_slider.observe(redraw, names="value")
    display(controls)
    redraw()


interactive_derivative_manipulation(df_derivative_marked)

## Optionaler Export

In [ ]:
if EXPORT_CLEAN_DERIVATIVE:
    export_dir = PROJECT_ROOT / EXPORT_DIR
    export_dir.mkdir(parents=True, exist_ok=True)
    clean_path = export_dir / "eps_real_derivative_clean.csv"
    outlier_path = export_dir / "eps_real_derivative_outliers.csv"
    df_derivative_clean.to_csv(clean_path, index=False)
    df_derivative_outliers.to_csv(outlier_path, index=False)
    print(f"Wrote {clean_path}")
    print(f"Wrote {outlier_path}")
else:
    print("Export disabled. Set EXPORT_CLEAN_DERIVATIVE = True to write CSV files.")